# Lab 3: Contextual Bandit-Based News Article Recommendation

**`Course`:** Reinforcement Learning Fundamentals  
**`Student Name`:**  
**`Roll Number`:**  
**`GitHub Branch`:** firstname_U20230xxx  

# Imports and Setup

In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from rlcmab_sampler import sampler


# Load Datasets

In [11]:
# Load datasets
news_df = pd.read_csv("data/news_articles.csv")
train_users = pd.read_csv("data/train_users.csv")
test_users = pd.read_csv("data/test_users.csv")

print(news_df.head())
print(train_users.head())


                                                link  \
0  https://www.huffpost.com/entry/covid-boosters-...   
1  https://www.huffpost.com/entry/american-airlin...   
2  https://www.huffpost.com/entry/funniest-tweets...   
3  https://www.huffpost.com/entry/funniest-parent...   
4  https://www.huffpost.com/entry/amy-cooper-lose...   

                                            headline   category  \
0  Over 4 Million Americans Roll Up Sleeves For O...  U.S. NEWS   
1  American Airlines Flyer Charged, Banned For Li...  U.S. NEWS   
2  23 Of The Funniest Tweets About Cats And Dogs ...     COMEDY   
3  The Funniest Tweets From Parents This Week (Se...  PARENTING   
4  Woman Who Called Cops On Black Bird-Watcher Lo...  U.S. NEWS   

                                   short_description               authors  \
0  Health experts said it is too early to predict...  Carla K. Johnson, AP   
1  He was subdued by passengers and crew when he ...        Mary Papenfuss   
2  "Until you have a dog y

## Data Preprocessing

In this section:
- Filter news articles to the 4 bandit arm categories (Entertainment, Education, Tech, Crime)
- Handle missing values (age imputation, dropping incomplete news rows)
- Encode categorical features (browser_version, region_code, subscriber)
- Prepare data for user classification

In [12]:
# ── News Articles Preprocessing ──────────────────────────────────────────────

TARGET_CATEGORIES = ["ENTERTAINMENT", "EDUCATION", "TECH", "CRIME"]

# Keep only the 4 bandit-arm categories
news_df = news_df[news_df["category"].isin(TARGET_CATEGORIES)].copy()

# Standardise category names to title-case (matches arm mapping table)
news_df["category"] = news_df["category"].str.title()

# Drop rows with missing headlines (6 total in the full dataset)
news_df = news_df.dropna(subset=["headline"])

# Fill remaining text NaNs so downstream code doesn't break
news_df["short_description"] = news_df["short_description"].fillna("")
news_df["authors"] = news_df["authors"].fillna("Unknown")

news_df = news_df.reset_index(drop=True)
print(f"News articles after filtering: {len(news_df)}")
print(news_df["category"].value_counts())

News articles after filtering: 24042
category
Entertainment    17362
Crime             3562
Tech              2104
Education         1014
Name: count, dtype: int64


In [13]:
# ── User Data Preprocessing ──────────────────────────────────────────────────

NON_FEATURE_COLS = ["user_id", "label"]

# Device/session noise columns with no logical link to user type
NOISE_COLS = [
    "screen_brightness", "battery_percentage", "network_jitter",
    "background_app_count", "session_inactivity_duration",
    "cart_abandonment_count",
    "browser_version",   # high-cardinality string, no user-type signal
    "region_code",       # near-zero overlap between train/test
]

def preprocess_users(df, scaler=None, fit=True):
    """Clean and encode user features.

    Parameters
    ----------
    df : pd.DataFrame
        Raw user dataframe (train or test).
    scaler : StandardScaler or None
        Pre-fitted scaler (reused at test time).
    fit : bool
        If True, fit scaler on this data; otherwise transform only.

    Returns
    -------
    X : np.ndarray
        Processed feature matrix.
    scaler : StandardScaler
        Fitted StandardScaler.
    """
    data = df.copy()

    # 1. Impute missing age with median
    age_median = data["age"].median()
    data["age"] = data["age"].fillna(age_median)

    # 2. Convert subscriber bool → int
    data["subscriber"] = data["subscriber"].astype(int)

    # 3. Drop non-feature and noisy columns
    cols_to_drop = [c for c in NON_FEATURE_COLS + NOISE_COLS if c in data.columns]
    data.drop(columns=cols_to_drop, inplace=True)

    # 4. Standard-scale all features
    if fit:
        scaler = StandardScaler()
        X = scaler.fit_transform(data)
    else:
        X = scaler.transform(data)

    return X, scaler


# ── Process training users ───────────────────────────────────────────────────

# Encode target labels
target_le = LabelEncoder()
y_all = target_le.fit_transform(train_users["label"])  # user_1→0, user_2→1, user_3→2

X_all, feat_scaler = preprocess_users(train_users, fit=True)

print(f"Training feature matrix shape: {X_all.shape}")
print(f"Target classes: {target_le.classes_}")

Training feature matrix shape: (2000, 31)
Target classes: ['user_1' 'user_2' 'user_3']
Feature columns after encoding: ['age', 'income', 'clicks', 'purchase_amount', 'session_duration', 'content_variety', 'engagement_score', 'num_transactions', 'avg_monthly_spend', 'avg_cart_value', 'browsing_depth', 'revisit_rate', 'scroll_activity', 'time_on_site', 'interaction_count', 'preferred_price_range', 'discount_usage_rate', 'wishlist_size', 'product_views', 'repeat_purchase_gap (days)', 'churn_risk_score', 'loyalty_index', 'screen_brightness', 'battery_percentage', 'cart_abandonment_count', 'background_app_count', 'session_inactivity_duration', 'network_jitter', 'subscriber', 'browser_major', 'region_prefix']


/var/folders/35/cxzv8jtx5tx6mzhvcx92s9mr0000gn/T/ipykernel_73600/2713489907.py:51: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = data.select_dtypes(include=["object"]).columns.tolist()


## User Classification

Train a classifier to predict the user category (`user_1`, `user_2`, `user_3`)
using an 80/20 train-validation split.  
This classifier serves as the **Context Detector** for the contextual bandit.


In [14]:
# ── Train / Validation Split (80/20) ─────────────────────────────────────────

X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

print(f"Train size: {X_train.shape[0]}, Validation size: {X_val.shape[0]}")

Train size: 1600, Validation size: 400


In [15]:
# ── Train Classifiers ────────────────────────────────────────────────────────

# 1. Decision Tree (baseline)
dt_clf = DecisionTreeClassifier(random_state=42)
dt_clf.fit(X_train, y_train)
dt_val_acc = accuracy_score(y_val, dt_clf.predict(X_val))

# 2. Logistic Regression (baseline)
lr_clf = LogisticRegression(max_iter=1000, random_state=42)
lr_clf.fit(X_train, y_train)
lr_val_acc = accuracy_score(y_val, lr_clf.predict(X_val))

# 3. Random Forest with GridSearchCV
rf_param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
}
rf_gs = GridSearchCV(
    RandomForestClassifier(random_state=42),
    rf_param_grid, cv=5, scoring="accuracy", n_jobs=-1,
)
rf_gs.fit(X_train, y_train)
rf_clf = rf_gs.best_estimator_
rf_val_acc = accuracy_score(y_val, rf_clf.predict(X_val))

# 4. Gradient Boosting with GridSearchCV
gb_param_grid = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1],
    "max_depth": [3, 5],
}
gb_gs = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    gb_param_grid, cv=5, scoring="accuracy", n_jobs=-1,
)
gb_gs.fit(X_train, y_train)
gb_clf = gb_gs.best_estimator_
gb_val_acc = accuracy_score(y_val, gb_clf.predict(X_val))

# ── Compare all models ───────────────────────────────────────────────────────
results = {
    "Decision Tree": (dt_clf, dt_val_acc),
    "Logistic Regression": (lr_clf, lr_val_acc),
    f"Random Forest {rf_gs.best_params_}": (rf_clf, rf_val_acc),
    f"Gradient Boosting {gb_gs.best_params_}": (gb_clf, gb_val_acc),
}

for name, (_, acc) in results.items():
    print(f"{name:<55s} Validation Accuracy: {acc:.4f}")

best_name, (best_clf, best_acc) = max(results.items(), key=lambda x: x[1][1])
print(f"\nSelected model: {best_name} ({best_acc:.4f})")

Decision Tree  — Validation Accuracy: 0.8100
Logistic Regression — Validation Accuracy: 0.8225

Selected model: Logistic Regression


In [16]:
# ── Classification Report on Validation Set ──────────────────────────────────

y_val_pred = best_clf.predict(X_val)

print(f"Classification Report ({best_name}):\n")
print(classification_report(
    y_val, y_val_pred, target_names=target_le.classes_
))

Classification Report (Logistic Regression):

              precision    recall  f1-score   support

      user_1       0.86      0.80      0.82       142
      user_2       0.91      0.81      0.86       142
      user_3       0.71      0.87      0.78       116

    accuracy                           0.82       400
   macro avg       0.83      0.83      0.82       400
weighted avg       0.83      0.82      0.82       400



In [17]:
# ── Classify Test Users (Context Detection) ──────────────────────────────────

X_test, _ = preprocess_users(test_users, scaler=feat_scaler, fit=False)

test_user_preds = best_clf.predict(X_test)
test_users["predicted_label"] = target_le.inverse_transform(test_user_preds)

# Context mapping: user_1 → 0, user_2 → 1, user_3 → 2
CONTEXT_MAP = {"user_1": 0, "user_2": 1, "user_3": 2}
test_users["context"] = test_users["predicted_label"].map(CONTEXT_MAP)

print("Test user context distribution:")
print(test_users["predicted_label"].value_counts())
print(f"\nTest feature matrix shape: {X_test.shape}")

Test user context distribution:
predicted_label
user_2    686
user_3    663
user_1    651
Name: count, dtype: int64

Test feature matrix shape: (2000, 31)


/var/folders/35/cxzv8jtx5tx6mzhvcx92s9mr0000gn/T/ipykernel_73600/2713489907.py:51: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = data.select_dtypes(include=["object"]).columns.tolist()


# `Contextual Bandit`

## Reward Sampler Initialization

The sampler is initialized using the student's roll number `i`.
Rewards are obtained using `sampler.sample(j)`.


## Arm Mapping

| Arm Index (j) | News Category | User Context |
|--------------|---------------|--------------|
| 0–3          | Entertainment, Education, Tech, Crime | User1 |
| 4–7          | Entertainment, Education, Tech, Crime | User2 |
| 8–11         | Entertainment, Education, Tech, Crime | User3 |

## Epsilon-Greedy Strategy

This section implements the epsilon-greedy contextual bandit algorithm.


## Upper Confidence Bound (UCB)

This section implements the UCB strategy for contextual bandits.

## SoftMax Strategy

This section implements the SoftMax strategy with temperature $ \tau = 1$.


## Reinforcement Learning Simulation

We simulate the bandit algorithms for $T = 10,000$ steps and record rewards.

P.S.: Change $T$ value as and if required.


## Results and Analysis

This section presents:
- Average Reward vs Time
- Hyperparameter comparisons
- Observations and discussion


## Final Observations

- Comparison of Epsilon-Greedy, UCB, and SoftMax
- Effect of hyperparameters
- Strengths and limitations of each approach
